In [1]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

In [3]:
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
# Exclude non-feature columns and potential other targets
NON_FEATURES = ['date', 'ticker', 'permno', 'shrout', 'prc'] 

# Identify numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Filter features
FEATURES = []
for c in numeric_cols:
    if c == TARGET:
        continue
    if c in NON_FEATURES:
        continue
    # Exclude other return/abnormal return columns to prevent leakage
    # Assuming targets start with 'ret' or 'ar_'
    if c.startswith('ret') or c.startswith('ar_'):
        continue
    FEATURES.append(c)

# Remove missing values
model_data = data[[TARGET] + FEATURES + ['date', 'ticker']].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Number of features: {len(FEATURES)}")
print(f"Features: {FEATURES}")

Sample size: 16,743,676
Target: f_cumret1
Number of features: 31
Features: ['net_sentiment', 'extreme_bullish_80', 'extreme_bullish_90', 'extreme_bearish_80', 'extreme_bearish_90', 'disagreement_index', 'raw_volume', 'log_volume', 'unique_user_count', 'volume_diff', 'log_volume_change', 'abn_volume_5d', 'abn_attention_std_5d', 'attention_surge_5d', 'abn_volume_21d', 'abn_attention_std_21d', 'attention_surge_21d', 'abn_volume_63d', 'abn_attention_std_63d', 'attention_surge_63d', 'abn_volume_250d', 'abn_attention_std_250d', 'attention_surge_250d', 'silence_gap_hours', 'relative_volume', 'attention_hhi', 'abnormal_sentiment_1d', 'abnormal_sentiment_5d', 'abnormal_sentiment_21d', 'abnormal_sentiment_63d', 'abnormal_sentiment_250d']


# In-Sample LASSO regression

In [5]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Normalize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit LASSO regression model with cross-validation for optimal alpha
# Using 5-fold CV and a range of alphas
# max_iter increased to 10000 to ensure convergence
lasso_model = LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=10000)
lasso_model.fit(X_scaled, y)

# Make predictions
y_pred = lasso_model.predict(X_scaled)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample LASSO Regression Results")
print("=" * 50)
print(f"Optimal alpha (regularization): {lasso_model.alpha_:.6f}")
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print("\nTop 10 Coefficients (on standardized features):")
coef_df = pd.DataFrame({'feature': FEATURES, 'coef': lasso_model.coef_})
coef_df['abs_coef'] = coef_df['coef'].abs()
print(coef_df.sort_values('abs_coef', ascending=False).head(10)[['feature', 'coef']])
print(f"\nNumber of non-zero coefficients: {(lasso_model.coef_ != 0).sum()} / {len(FEATURES)}")
print(f"Intercept: {lasso_model.intercept_:.6f}")

In-Sample LASSO Regression Results
Optimal alpha (regularization): 0.000141
R-squared: 0.000068
RMSE: 0.047094
MSE: 0.002218

Top 10 Coefficients (on standardized features):
                   feature      coef
7               log_volume -0.000179
24         relative_volume -0.000106
29  abnormal_sentiment_63d  0.000036
9              volume_diff -0.000015
14          abn_volume_21d -0.000002
23       silence_gap_hours  0.000000
19     attention_surge_63d -0.000000
20         abn_volume_250d -0.000000
21  abn_attention_std_250d -0.000000
22    attention_surge_250d -0.000000

Number of non-zero coefficients: 5 / 31
Intercept: 0.000496


# OOS predictions

In [6]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year (in days) for rolling window
WINDOW_21 = 21    # One trading month (in days) for rolling window
MAX_ITER = 10000  # Increased iterations for convergence

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Create a year-month column for grouping
model_data['year_month'] = model_data['date'].dt.to_period('M')

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")

# Initialize storage for predictions and selected alphas
predictions_expanding = []
predictions_rolling_252 = []
predictions_rolling_21 = []
alphas_expanding = []
alphas_rolling_252 = []
alphas_rolling_21 = []
nonzero_coefs_expanding = []
nonzero_coefs_rolling_252 = []
nonzero_coefs_rolling_21 = []

# Loop through OOS months (train once per month)
for month_idx, pred_month in enumerate(oos_months):
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]

    if len(month_dates) == 0:
        continue

    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)

    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        continue
    last_train_date = train_dates[-1]

    # 1. EXPANDING WINDOW: Train on all data up to end of previous month
    train_mask_exp = model_data['date'] <= last_train_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]

    if len(X_train_exp) > 0:
        # Normalize features
        scaler_exp = StandardScaler()
        X_train_exp_scaled = scaler_exp.fit_transform(X_train_exp)
        
        # Fit LASSO with CV
        lasso_exp = LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
        lasso_exp.fit(X_train_exp_scaled, y_train_exp)
        alphas_expanding.append({'month': pred_month, 'alpha': lasso_exp.alpha_})
        nonzero_coefs_expanding.append({'month': pred_month, 'n_nonzero': (lasso_exp.coef_ != 0).sum()})

        # Use this model to predict for all days in the month
        for pred_date in month_dates:
            test_mask = model_data['date'] == pred_date
            X_test = model_data.loc[test_mask, FEATURES]

            if len(X_test) == 0:
                continue

            # Transform test features using the same scaler
            X_test_scaled = scaler_exp.transform(X_test)
            
            test_indices = model_data.index[test_mask]
            y_pred_exp = lasso_exp.predict(X_test_scaled)

            for idx, pred in zip(test_indices, y_pred_exp):
                predictions_expanding.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_expanding': pred
                })

    # 2. ROLLING 252-DAY WINDOW: Train on last 252 trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx >= WINDOW_252:
        start_date_252 = unique_dates[last_train_date_idx - WINDOW_252 + 1]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] <= last_train_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]

        if len(X_train_252) > 0:
            # Normalize features
            scaler_252 = StandardScaler()
            X_train_252_scaled = scaler_252.fit_transform(X_train_252)
            
            # Fit LASSO with CV
            lasso_252 = LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
            lasso_252.fit(X_train_252_scaled, y_train_252)
            alphas_rolling_252.append({'month': pred_month, 'alpha': lasso_252.alpha_})
            nonzero_coefs_rolling_252.append({'month': pred_month, 'n_nonzero': (lasso_252.coef_ != 0).sum()})

            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]

                if len(X_test) == 0:
                    continue

                # Transform test features using the same scaler
                X_test_scaled = scaler_252.transform(X_test)
                
                test_indices = model_data.index[test_mask]
                y_pred_252 = lasso_252.predict(X_test_scaled)

                for idx, pred in zip(test_indices, y_pred_252):
                    predictions_rolling_252.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_252': pred
                    })

    # 3. ROLLING 21-DAY WINDOW: Train on last 21 trading days before the month
    if last_train_date_idx >= WINDOW_21:
        start_date_21 = unique_dates[last_train_date_idx - WINDOW_21 + 1]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] <= last_train_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]

        if len(X_train_21) > 0:
            # Normalize features
            scaler_21 = StandardScaler()
            X_train_21_scaled = scaler_21.fit_transform(X_train_21)
            
            # Fit LASSO with CV
            lasso_21 = LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
            lasso_21.fit(X_train_21_scaled, y_train_21)
            alphas_rolling_21.append({'month': pred_month, 'alpha': lasso_21.alpha_})
            nonzero_coefs_rolling_21.append({'month': pred_month, 'n_nonzero': (lasso_21.coef_ != 0).sum()})

            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]

                if len(X_test) == 0:
                    continue

                # Transform test features using the same scaler
                X_test_scaled = scaler_21.transform(X_test)
                
                test_indices = model_data.index[test_mask]
                y_pred_21 = lasso_21.predict(X_test_scaled)

                for idx, pred in zip(test_indices, y_pred_21):
                    predictions_rolling_21.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_21': pred
                    })

    # Progress update
    if (month_idx + 1) % 12 == 0:
        print(f"Processed {month_idx + 1}/{len(oos_months)} months ({100 * (month_idx + 1) / len(oos_months):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_expanding):,}")
print(f"Rolling 252-day predictions: {len(predictions_rolling_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_rolling_21):,}")

Training window: 2010-01-04 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-12-30
Number of OOS dates: 3,269
Number of OOS months: 156


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.282e-02, tolerance: 7.349e-03
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.010e-02, tolerance: 7.535e-03
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or c

Processed 12/156 months (7.7%)


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.364e-02, tolerance: 6.523e-03
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.776e-02, tolerance: 7.920e-03
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or c

Processed 24/156 months (15.4%)
Processed 36/156 months (23.1%)
Processed 48/156 months (30.8%)
Processed 60/156 months (38.5%)
Processed 72/156 months (46.2%)
Processed 84/156 months (53.8%)
Processed 96/156 months (61.5%)
Processed 108/156 months (69.2%)
Processed 120/156 months (76.9%)
Processed 132/156 months (84.6%)


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.067e-02, tolerance: 3.384e-02
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.838e-02, tolerance: 3.384e-02
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or c

Processed 144/156 months (92.3%)


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.673e-02, tolerance: 3.172e-02
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.545e-01, tolerance: 3.133e-02
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or c

Processed 156/156 months (100.0%)

Completed.
Expanding window predictions: 14,545,760
Rolling 252-day predictions: 14,545,760
Rolling 21-day predictions: 14,545,760


In [7]:
# Summary of selected alpha values and non-zero coefficients
print("Selected Alpha (Regularization Parameter) Summary")
print("=" * 50)

if alphas_expanding:
    alphas_exp_df = pd.DataFrame(alphas_expanding)
    nonzero_exp_df = pd.DataFrame(nonzero_coefs_expanding)
    print(f"\nExpanding Window:")
    print(f"  Mean alpha: {alphas_exp_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_exp_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_exp_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_exp_df['alpha'].max():.6f}")
    print(f"  Mean non-zero coefficients: {nonzero_exp_df['n_nonzero'].mean():.1f} / {len(FEATURES)}")

if alphas_rolling_252:
    alphas_252_df = pd.DataFrame(alphas_rolling_252)
    nonzero_252_df = pd.DataFrame(nonzero_coefs_rolling_252)
    print(f"\nRolling 252-day Window:")
    print(f"  Mean alpha: {alphas_252_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_252_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_252_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_252_df['alpha'].max():.6f}")
    print(f"  Mean non-zero coefficients: {nonzero_252_df['n_nonzero'].mean():.1f} / {len(FEATURES)}")

if alphas_rolling_21:
    alphas_21_df = pd.DataFrame(alphas_rolling_21)
    nonzero_21_df = pd.DataFrame(nonzero_coefs_rolling_21)
    print(f"\nRolling 21-day Window:")
    print(f"  Mean alpha: {alphas_21_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_21_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_21_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_21_df['alpha'].max():.6f}")
    print(f"  Mean non-zero coefficients: {nonzero_21_df['n_nonzero'].mean():.1f} / {len(FEATURES)}")

Selected Alpha (Regularization Parameter) Summary

Expanding Window:
  Mean alpha: 0.000101
  Std alpha: 0.000102
  Min alpha: 0.000001
  Max alpha: 0.000362
  Mean non-zero coefficients: 6.5 / 31

Rolling 252-day Window:
  Mean alpha: 0.000158
  Std alpha: 0.000177
  Min alpha: 0.000001
  Max alpha: 0.000968
  Mean non-zero coefficients: 7.7 / 31

Rolling 21-day Window:
  Mean alpha: 0.000563
  Std alpha: 0.000464
  Min alpha: 0.000000
  Max alpha: 0.002273
  Mean non-zero coefficients: 3.3 / 31


In [8]:
# Convert predictions to DataFrames and merge
df_expanding = pd.DataFrame(predictions_expanding)
df_rolling_252 = pd.DataFrame(predictions_rolling_252)
df_rolling_21 = pd.DataFrame(predictions_rolling_21)

# Merge all predictions together
predictions_df = df_expanding.merge(
    df_rolling_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_rolling_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"\nNon-null predictions by window type:")
print(f"  Expanding: {predictions_df['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_df['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_df['pred_rolling_21'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 14,545,760

Non-null predictions by window type:
  Expanding: 14,545,760
  Rolling 252-day: 14,545,760
  Rolling 21-day: 14,545,760

First 10 predictions:
        date  permno ticker    index  pred_expanding  pred_rolling_252  \
0 2012-01-03   87432      A  2879304        0.000495         -0.000204   
1 2012-01-03   24643     AA  2377768        0.000495         -0.000204   
2 2012-01-03   12479    AAC  2266267        0.000495         -0.000204   
3 2012-01-03   90020   AACC  2994307        0.000495         -0.000204   
4 2012-01-03   15580   AAME  2343833        0.000495         -0.000204   
5 2012-01-03   10517    AAN  2211052        0.000495         -0.000204   
6 2012-01-03   76868   AAON  2576068        0.000495         -0.000204   
7 2012-01-03   89217    AAP  2947508        0.000495         -0.000204   
8 2012-01-03   14593   AAPL  2339391        0.000125         -0.000204   
9 2012-01-03   90854   AATI  3056541        0.000495         -0.0

In [9]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_lasso_all_features.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
if os.path.exists(OUTPUT_FILE):
    print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")

Predictions saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_lasso_all_features.pkl
File size: 757.55 MB
Shape: (14545760, 7)
